# Multilingual RAG — Indian Government Policy Q&A

Answer questions about Indian government policies in English, Hindi, or Tamil.
The pipeline detects the query language, retrieves relevant documents using
cross-lingual semantic search, generates an answer with `sarvam-m`, then
translates the answer back to the user's language.

## Pipeline

```
User query (any supported language)
        |
        v
detect_query_language()    <- Sarvam identify_language API
        |
        v
retrieve_documents()       <- sentence-transformers + FAISS
        |  query embedded directly in its original language
        |  paraphrase-multilingual-mpnet-base-v2
        v
generate_answer()          <- sarvam-m (chat.completions)
        |  context: content_english from retrieved docs
        |  query translated to English for the LLM prompt
        v
translate_text()           <- Sarvam translate API
        |  skipped when detected language is en-IN
        v
Answer in the user's language
```

## Supported Languages

| Code | Language |
|:-----|:---------|
| en-IN | English (India) |
| hi-IN | Hindi |
| ta-IN | Tamil |

In [ ]:
%pip install sarvamai>=0.1.26 sentence-transformers>=2.2.2 faiss-cpu>=1.7.4 python-dotenv>=1.0.0 pandas>=1.5.0 numpy>=1.24.0

## Setup and API Key

Load environment variables and initialise the Sarvam AI client.
Set `SARVAM_API_KEY` in a `.env` file copied from `.env.example`.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sarvamai import SarvamAI
from sentence_transformers import SentenceTransformer

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
print("Sarvam AI client initialised.")

## Load Multilingual Documents

Load the sample corpus from `sample_data/sample_docs.json`.
Each document has a `content` field in its original language for embedding
and a `content_english` field used as context for the LLM.

In [ ]:
def load_documents(json_path: str) -> pd.DataFrame:
    """Load sample documents from a JSON file into a DataFrame.

    Args:
        json_path: Path to the JSON file containing a list of document dicts.

    Returns:
        DataFrame with one row per document.
    """
    path = Path(json_path)
    with path.open(encoding="utf-8") as fh:
        docs = json.load(fh)
    return pd.DataFrame(docs)

In [ ]:
DOCS_PATH = "sample_data/sample_docs.json"
df = load_documents(DOCS_PATH)

print(f"Loaded {len(df)} documents")
print(f"Languages : {sorted(df['language'].unique())}")
print(f"Topics    : {sorted(df['topic'].unique())}")
df[["id", "language", "topic", "title"]]

## Build Embedding Index

Embed every document using a multilingual sentence-transformer model.
The model maps semantically equivalent text across languages to nearby
vectors, enabling cross-lingual retrieval without pre-translating the query.

Model: `paraphrase-multilingual-mpnet-base-v2`
- Dimensions: 768
- Languages: 50+
- Download size: ~420 MB on first run (cached locally after that)

In [ ]:
def embed_documents(
    df: pd.DataFrame,
    model_name: str = "paraphrase-multilingual-mpnet-base-v2",
) -> tuple[Any, np.ndarray]:
    """Embed all documents using a multilingual sentence-transformer model.

    Documents are embedded from their original-language `content` field.
    Embeddings are L2-normalised so that dot product equals cosine similarity.

    Args:
        df: DataFrame containing a `content` column.
        model_name: HuggingFace model identifier for SentenceTransformer.

    Returns:
        Tuple of (loaded model, float32 embeddings array of shape (n, dim)).
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        df["content"].tolist(),
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return model, np.array(embeddings, dtype="float32")

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> Any:
    """Build a FAISS IndexFlatIP index from L2-normalised embeddings.

    IndexFlatIP computes inner product, which equals cosine similarity
    when vectors are L2-normalised. No training step is required.

    Args:
        embeddings: float32 ndarray of shape (n, dim), L2-normalised.

    Returns:
        Populated FAISS IndexFlatIP index ready for search.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

In [ ]:
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

embedding_model, doc_embeddings = embed_documents(df, EMBEDDING_MODEL_NAME)
faiss_index = build_faiss_index(doc_embeddings)

print(f"Model          : {EMBEDDING_MODEL_NAME}")
print(f"Embedding dims : {doc_embeddings.shape[1]}")
print(f"Docs indexed   : {faiss_index.ntotal}")